# 🎓 Tutor da CPA — chatbot com IA (Gemini)

Tutor que explica temas da prova **CPA (ANBIMA)** e gera uma questão de múltipla escolha, usando **só os números de uma tabela oficial** passada nas instruções, para reduzir respostas inventadas.

> Para rodar: no Colab, cadastre sua chave do Gemini em **Secrets** (ícone 🔑) com o nome `API_KEY`.

## 1. Carregar a chave da API (sem expor no código)

In [ ]:
from google.colab import userdata
chave = userdata.get("API_KEY")
print("Chave encontrada!" if chave else "Não achei a chave")

Chave encontrada!


## 2. Instalar o SDK do Gemini

In [ ]:
# Célula nova – instalar a biblioteca do Gemini
!pip install -q google-genai

## 3. Teste rápido: pergunta direta ao modelo

In [ ]:
from google.colab import userdata
from google import genai

cliente = genai.Client(api_key=userdata.get("API_KEY"))

resposta = cliente.models.generate_content(
    model="gemini-3.8-flash",
    contents="Explique em 3 frases o que é come-cotas, para quem vai fazer a CPA."
)
print(resposta.text)

1. O **come-cotas** é a antecipação semestral do Imposto de Renda que incide sobre fundos de Renda Fixa, Cambiais e Multimercado, ocorrendo sempre no último dia útil dos meses de **maio e novembro**.

2. A cobrança é feita por meio da **redução da quantidade de cotas** do investidor, aplicando-se a alíquota mínima da categoria: **20%** para fundos de Curto Prazo e **15%** para fundos de Longo Prazo.

3. Para a prova da CPA, é essencial lembrar que ele **não incide sobre Fundos de Ações** e que, no momento do resgate, o investidor paga apenas a alíquota complementar, caso o prazo final exija uma taxa maior.


## 4. Instruções do tutor (system prompt)

Regras de formato + tabela oficial. O modelo é orientado a usar **somente** esses números e dizer "não sei" quando não encontrar a resposta.

In [ ]:
from google.genai import types

INSTRUCOES = """
Você é um tutor da prova CPA da ANBIMA. Fale em português do Brasil, de forma simples.
Regras:
1. Explique o tema em no máximo 4 frases.
2. Depois faça UMA pergunta de múltipla escolha (A, B, C, D) sobre o tema.
3. Use SOMENTE os números da tabela abaixo. Se não souber, diga que não sabe.

TABELA OFICIAL
- IR renda fixa (tabela regressiva): até 180 dias 22,5%; 181 a 360 dias 20%; 361 a 720 dias 17,5%; acima de 720 dias 15%.
- Come-cotas: maio e novembro; 15% longo prazo; 20% curto prazo; não existe em fundo de ações.
"""
print("Manual do tutor carregado!")

Manual do tutor carregado!


## 5. Tutor em ação

`temperature=0.2` para respostas mais consistentes.

In [ ]:
resposta = cliente.models.generate_content(
    model="gemini-3.8-flash",
    contents="Quero estudar come-cotas",
    config=types.GenerateContentConfig(
        system_instruction=INSTRUCOES,
        temperature=0.2,
    ),
)
print(resposta.text)

O come-cotas é uma antecipação semestral do Imposto de Renda em fundos de investimento que reduz a quantidade de cotas que você possui. Essa cobrança acontece sempre nos meses de maio e novembro. A alíquota é de 15% para fundos de longo prazo e de 20% para fundos de curto prazo. Além disso, o come-cotas não existe em fundo de ações.

---

**Pergunta:**
Qual é a alíquota do come-cotas para fundos de longo prazo e em qual tipo de fundo ele não existe?

A) 15% e não existe em fundo de ações.  
B) 20% e não existe em fundo de ações.  
C) 17,5% e não existe em fundo de ações.  
D) 22,5% e não existe em fundo de ações.


## Próximos passos

- **RAG:** carregar o material completo da CPA, quebrar em trechos, gerar embeddings e buscar só os trechos relevantes para cada pergunta.
- Corrigir a resposta do aluno e explicar o erro.
- Interface simples (Streamlit ou Gradio).